In [1]:
from env import SimpleARGEnvironment
from utils import load_sequences

In [2]:
Ne = 10000
r_per_bp = 2e-8
mu_per_bp = 2e-8

dataset_path="validation/fasta/sim_l25kb_0.fa"

sequences = load_sequences(dataset_path)

In [3]:
env = SimpleARGEnvironment(
    num_sequences=len(sequences),
    population_size=Ne,
    recombination_rate=r_per_bp,
    mutation_rate=mu_per_bp,
    sequences=sequences,
    seed=7,
    bp_per_blocks=1
)

episodes = 2

In [4]:
from tb_gfn import TBGFlowNetGenerator

generator = TBGFlowNetGenerator(env, 1)
model = generator.arg_model

In [5]:
import torch
import numpy as np

states = [env.get_initial_state() for _ in range(episodes)]

input_dict = env.prepare_state_rollout_inputs(
    states,
)

# ret = generator(input_dict)
# # 


In [6]:
ret = model(input_dict)

In [7]:
ret

(tensor([-15.4091, -15.6388], grad_fn=<AddBackward0>),
 tensor([2.0320e-07, 1.6149e-07], grad_fn=<ExpBackward0>),
 [RecombinationChoice(active_lineage_i=1, material_count=25000, span_start=0, span_end=24999, time_action=19, breakpoint=tensor([23683])),
  RecombinationChoice(active_lineage_i=2, material_count=25000, span_start=0, span_end=24999, time_action=6, breakpoint=tensor([1170]))])

In [ ]:
lineage_reps, summary_reps, lineage_seq_features, batch_active_lineage_counts = model._encode_states(states)

all_candidate_actions = input_dict.get("input_actions")


logits = model._score_candidates(
    all_candidate_actions,
    lineage_reps,
    summary_reps,
)

choosen_action_indices = model.sample(logits)
choosen_actions = [all_candidate_actions[batch_idx][action_idx] for batch_idx, action_idx in enumerate(choosen_action_indices)]

# Compute log pf for action scorer (policy) selection
log_action_pf = model.compute_log_path_pf(logits, choosen_action_indices)

In [ ]:
from env import RecombinationChoice
log_breakpoint_pf = []
for idx, (chosen_action, action_idx) in enumerate(zip(choosen_actions, choosen_action_indices)):
    if isinstance(chosen_action, RecombinationChoice):
        # Get the candidate action before we changed the breakpoint
        original_action = all_candidate_actions[idx][action_idx]
        # Get the sequence array for the relevant lineage ## TODO: it cannot be active lineage i, it should be total material 
        seq_array = env.seq_arrays[original_action.active_lineage_i].unsqueeze(0)  # shape [1, L]
        # Get logits for this lineage sequence
        bp_logits = model.breakpoint_scorer(seq_array)

        # The chosen breakpoint for this action
        bp_action = chosen_action.breakpoint
        # Logits index is breakpoint-1 (since k==i+1 convention)
        log_p_bp = model.logsoftmax(bp_logits)
        log_breakpoint_pf.append(log_p_bp)
    else:
        log_breakpoint_pf.append(torch.zeros((), device=model.device))
log_breakpoint_pf = torch.stack(log_breakpoint_pf)


In [ ]:
log_breakpoint_pf

In [ ]:
kl = model.logsoftmax(bp_logits)[0, breakpoint]

In [ ]:
batch_idx = torch.arange(log_breakpoint_pf.shape[0], device=logits.device)
batch_idx
log_p = model.logsoftmax(log_breakpoint_pf)
log_p

In [ ]:

log_breakpoint_pf.shape, log_p

In [ ]:
np.exp(-10.1769)

In [ ]:
logits = model._score_candidates(
            all_candidate_actions,
            lineage_reps,
            summary_reps,
        )

In [ ]:
logits.shape

In [ ]:
from torch.distributions import Categorical
Categorical(logits=logits).sample()

In [ ]:
logits = model._score_candidates(
    input_actions,
    lineage_reps,
    summary_reps,
    lineage_seq_features,
    states,
    batch_active_lineage_counts,
)

In [ ]:
logits.shape

In [ ]:
from torch.distributions import Categorical
l = Categorical(logits=logits).sample()

In [ ]:
choosen_actions = [input_actions[batch_idx][action_idx] for batch_idx, action_idx in enumerate(l)]

In [ ]:
choosen_actions[1]

In [ ]:
valid_breakpoints = model._action_feature_breakpoints(choosen_actions[1])

In [ ]:
len(valid_breakpoints)

In [ ]:
for batch_idx, actions in enumerate(input_actions):
    state_action_features  = model._batched_action_features(
                actions,
                batch_idx,
                lineage_reps,
                summary_reps
            )
    break

In [ ]:
state_action_features.shape

In [ ]:
feat_dim = model.seq_embedding.out_features * 6
print(feat_dim)
features = lineage_reps.new_zeros(1, len(actions), feat_dim)

In [ ]:
features[0, :len(actions)].shape


In [ ]:
len(actions)

In [ ]:
## Encode states

active_counts = [len(state.active_lineages) for state in active_states]
max_active = max(active_counts)
sequence_length = env.seq_arrays.shape[1] ## sequence_length

## torch.Size([1, 8, 25000, 4])
lineage_features = env.seq_arrays.new_zeros(1,  max_active,  sequence_length,  4)


In [ ]:
lineage_features = env.seq_arrays.new_zeros(
            1,
            max_active,
            25000,
            4,
        )